In [5]:
import os
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.utils.data as DataLoader
from PIL import Image

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# import the FashionMNIST dataset from ./data directory
train_dataset = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 60000
Test dataset size: 10000


In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)# in_channels = 1 for grayscaled images, out_channels = number of filters, kernel_size = size of the filter, stride = step size, padding = 0 for no padding
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128) # 32 filters of size 7x7 after pooling
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x


device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN().to(device)
print(model.parameters())

<generator object Module.parameters at 0x0000024E89274AC0>


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN().to(device)

model.load_state_dict(torch.load("./saved_models/best_model.pth", weights_only=True))

<All keys matched successfully>

In [8]:
model.eval()

inference_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), # Ensure it's 1-channel (grayscale)
    transforms.Resize((28, 28)),                 # Match training image dimensions
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))         # Match training normalization
])

def predict_image(img_path, model, transform, device, class_names):
    img = Image.open(img_path)
    img_tensor = transform(img)
    image = img_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        raw_output = model(image)
        _, predicted_class = torch.max(raw_output, 1)
        probs = torch.nn.functional.softmax(raw_output, dim=1)
        print(f"probabilities: {probs}")
        confidence = probs[0][predicted_class].item() * 100
        print(f"confidence: {confidence:.2f}%")
    predicted_class_name = class_names[predicted_class.item()]
    return predicted_class_name, confidence


In [10]:
fashion_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

test_image_path = "./Images/667626_18933d713e.jpg"
predicted_class, confidence = predict_image(test_image_path, model, inference_transform, device, fashion_classes)
print(f"Predicted class: {predicted_class}, Confidence: {confidence:.2f}%")

probabilities: tensor([[0.0635, 0.0481, 0.0019, 0.0039, 0.0034, 0.1338, 0.0925, 0.0024, 0.5626,
         0.0878]])
confidence: 56.26%
Predicted class: Bag, Confidence: 56.26%
